# 📖 Notebook 1: Routing & Load Balancing

This notebook teaches you the **core job** of an API Gateway: routing requests to the right backend service and balancing traffic across multiple instances.

We'll go through three approaches:
- 🚫 **BAD**: Clients call each service directly (tight coupling)
- ✅ **BETTER**: A basic reverse proxy (single entry point)
- 🏆 **BEST**: Full API gateway with path-based routing, load balancing, and health checks

## Learning Objectives

By the end of this notebook, you'll understand:
- Why direct client-to-service calls are problematic
- What a reverse proxy does and why it helps
- How path-based routing directs traffic to different services
- How load balancing distributes requests across service instances
- How health checks keep the system reliable

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 05-microservices/api-gateway
docker-compose up -d --build
```

This starts:
- **2 User Service instances** (ports 5001, 5003) — for load balancing
- **1 Order Service instance** (port 5002)
- **nginx API Gateway** (port 8080) — the single entry point
- **Redis** (port 6380) — used in Notebook 2

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import json

# Helper to pretty-print JSON responses
def show(response):
    """Print HTTP status and JSON body in a readable format."""
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

# Quick health check — make sure everything is running
try:
    r = requests.get("http://localhost:5001/health", timeout=3)
    print(f"✅ User Service 1: {r.json()['status']}")
except Exception as e:
    print(f"❌ User Service 1 not running: {e}")

try:
    r = requests.get("http://localhost:5003/health", timeout=3)
    print(f"✅ User Service 2: {r.json()['status']}")
except Exception as e:
    print(f"❌ User Service 2 not running: {e}")

try:
    r = requests.get("http://localhost:5002/health", timeout=3)
    print(f"✅ Order Service:  {r.json()['status']}")
except Exception as e:
    print(f"❌ Order Service not running: {e}")

try:
    r = requests.get("http://localhost:8080/health", timeout=3)
    print(f"✅ API Gateway:    {r.json()['status']}")
except Exception as e:
    print(f"❌ API Gateway not running: {e}")
    print("   Run: cd 05-microservices/api-gateway && docker-compose up -d --build")

---

## 🚫 BAD: Direct Client-to-Service Calls

Without an API gateway, clients must talk to each service directly. This means:
- The client needs to know **every service's address and port**
- If a service moves or scales, **every client must be updated**
- There's **no centralized place** for auth, rate limiting, or logging

```
┌──────────┐     ┌───────────────────┐
│  Client   │────▶│ User Service :5001 │
│           │     └───────────────────┘
│           │     ┌───────────────────┐
│           │────▶│ User Service :5003 │   ← Client must know BOTH instances!
│           │     └───────────────────┘
│           │     ┌───────────────────┐
│           │────▶│ Order Service:5002 │
└──────────┘     └───────────────────┘
```

Let's see what this looks like in code:

In [ ]:
# BAD: The client needs to know each service's address and port.
# If you add a new service or move one to a different server,
# you'd have to update EVERY client.

# Client configuration — imagine maintaining this for 50 services!
SERVICE_REGISTRY = {
    "user_service_1": "http://localhost:5001",
    "user_service_2": "http://localhost:5003",
    "order_service":  "http://localhost:5002",
}

print("🚫 BAD: Client calls each service directly")
print("=" * 55)
print()

# Call the user service directly
print("📡 Calling User Service (instance 1) at port 5001:")
r = requests.get(f"{SERVICE_REGISTRY['user_service_1']}/users/1")
show(r)
print()

# Call the order service directly
print("📡 Calling Order Service at port 5002:")
r = requests.get(f"{SERVICE_REGISTRY['order_service']}/orders/101")
show(r)

In [ ]:
# The client also has to implement its own load balancing!
# This is complex, error-prone, and duplicated across every client.

import random

def bad_client_side_load_balance(service_urls: list, path: str):
    """BAD: Client picks a random instance and hopes it's healthy."""
    url = random.choice(service_urls)
    return requests.get(f"{url}{path}")

user_service_urls = [
    "http://localhost:5001",
    "http://localhost:5003",
]

print("🚫 BAD: Client-side load balancing")
print("The client randomly picks an instance — no health checking!")
print()

for i in range(6):
    r = bad_client_side_load_balance(user_service_urls, "/users")
    data = r.json()
    print(f"  Request {i+1}: routed to {data['served_by']}")

print()
print("⚠️  Problems with this approach:")
print("   1. Client needs to know ALL service addresses")
print("   2. Client must implement load balancing logic")
print("   3. No health checking — might send traffic to dead instances")
print("   4. Every client (web, mobile, CLI) duplicates this logic")

---

## ✅ BETTER: Basic Reverse Proxy

A **reverse proxy** sits between clients and backend services. Clients send all requests to **one address**, and the proxy forwards them to the right place.

```
┌──────────┐     ┌───────────────┐     ┌───────────────────┐
│  Client   │────▶│ Reverse Proxy │────▶│ Backend Service    │
│           │     │  (nginx:8080) │     │                   │
└──────────┘     └───────────────┘     └───────────────────┘
```

This is already a **huge improvement**:
- Clients only need to know **one address** (the proxy)
- Backend services can move or scale without client changes
- You have a centralized place for logging and monitoring

A simple nginx reverse proxy config looks like this:

```nginx
# BETTER: Simple reverse proxy — one entry point, but no smart routing
server {
    listen 80;
    
    # Everything goes to one backend
    location / {
        proxy_pass http://backend-service:5000;
    }
}
```

But this only forwards to **one service**. What if you have many services?  
That's where an API gateway's **path-based routing** comes in.

In [ ]:
# BETTER: Client only needs one address — the gateway
# Compare this to the BAD approach above where we needed 3 different URLs

GATEWAY = "http://localhost:8080"

print("✅ BETTER: All requests go through the gateway")
print(f"   Client only knows: {GATEWAY}")
print()

# Even though we have 3 service instances behind the scenes,
# the client just calls the gateway
print("📡 Getting users through the gateway:")
r = requests.get(f"{GATEWAY}/api/users/1")
show(r)
print()

print("📡 Getting orders through the gateway:")
r = requests.get(f"{GATEWAY}/api/orders/101")
show(r)
print()

print("💡 Notice: The client doesn't know (or care) that users and orders")
print("   are served by completely different services on different ports!")

---

## 🏆 BEST: API Gateway with Path-Based Routing, Load Balancing & Health Checks

Our nginx API gateway does three things at once:

### 1. Path-Based Routing
The gateway reads the URL path and routes to the correct backend:

```
/api/users/*  ──▶  User Service   (upstream: user-service-1, user-service-2)
/api/orders/* ──▶  Order Service  (upstream: order-service)
```

Here's the relevant nginx config:

```nginx
# Route /api/users to the user service backend
location /api/users {
    proxy_pass http://user_backend/users;
}

# Route /api/orders to the order service backend
location /api/orders {
    proxy_pass http://order_backend/orders;
}
```

### 2. Load Balancing
The `user_backend` upstream has two servers — nginx distributes requests between them:

```nginx
upstream user_backend {
    server user-service-1:5000;   # Instance 1
    server user-service-2:5000;   # Instance 2
}
```

### 3. Health Checks
Each service exposes a `/health` endpoint. The gateway depends on healthy backends before starting.

Let's see all of this in action!

In [ ]:
# BEST: Path-based routing — one URL, multiple services

GATEWAY = "http://localhost:8080"

print("🏆 BEST: Path-Based Routing")
print("=" * 55)
print()
print("The gateway reads the URL path and routes to the correct service.")
print("The client doesn't know which service handles which path.")
print()

# All these go through the SAME gateway on port 8080
routes = [
    ("/api/users",      "→ routed to User Service"),
    ("/api/users/1",    "→ routed to User Service"),
    ("/api/orders",     "→ routed to Order Service"),
    ("/api/orders/101", "→ routed to Order Service"),
]

for path, description in routes:
    r = requests.get(f"{GATEWAY}{path}")
    data = r.json()
    served = data.get("served_by", "unknown")
    print(f"  GET {path:<20} {description}  (served by: {served})")

print()
print("💡 Same gateway URL, different backend services!")
print("   Adding a new service is just a new nginx location block.")

In [ ]:
# BEST: Load Balancing — requests are distributed across instances

print("🏆 BEST: Load Balancing")
print("=" * 55)
print()
print("nginx uses round-robin by default: it alternates between instances.")
print("Watch the 'served_by' field change between requests!")
print()

instance_count = {}

for i in range(10):
    r = requests.get(f"{GATEWAY}/api/users")
    served_by = r.json()["served_by"]
    instance_count[served_by] = instance_count.get(served_by, 0) + 1
    print(f"  Request {i+1:>2}: handled by {served_by}")

print()
print("📊 Distribution:")
for instance, count in sorted(instance_count.items()):
    bar = "█" * count
    print(f"  {instance}: {count} requests  {bar}")

print()
print("💡 nginx spreads the load evenly — no single instance gets overwhelmed.")
print("   Other strategies: 'least_conn' (send to least busy), 'ip_hash' (sticky sessions).")

In [ ]:
# BEST: Health Checks — the gateway knows which backends are alive

print("🏆 BEST: Health Checks")
print("=" * 55)
print()

# Check gateway health
r = requests.get(f"{GATEWAY}/health")
print("Gateway health:")
show(r)
print()

# Check individual service health (through direct ports)
services = [
    ("User Service 1", "http://localhost:5001/health"),
    ("User Service 2", "http://localhost:5003/health"),
    ("Order Service",  "http://localhost:5002/health"),
]

print("Backend service health:")
for name, url in services:
    try:
        r = requests.get(url, timeout=2)
        status = r.json()["status"]
        instance = r.json()["instance"]
        print(f"  ✅ {name}: {status} (instance {instance})")
    except Exception as e:
        print(f"  ❌ {name}: DOWN ({e})")

print()
print("💡 In production, the gateway automatically removes unhealthy backends.")
print("   If user-service-1 crashes, ALL traffic goes to user-service-2.")
print("   The client never knows anything went wrong.")

In [ ]:
# Let's compare: BAD (direct) vs BEST (gateway) side by side

import time

print("📊 Comparison: Direct Calls vs API Gateway")
print("=" * 60)
print()

# BAD: Direct calls — client must know all addresses
print("🚫 BAD: Direct calls (client manages service discovery)")
print(f"   URLs the client needs to know: 3")
print(f"   - http://localhost:5001 (user instance 1)")
print(f"   - http://localhost:5003 (user instance 2)")
print(f"   - http://localhost:5002 (order service)")
print(f"   Load balancing: client-side (manual)")
print(f"   Auth/Rate limiting: each service implements its own")
print()

# BEST: Gateway — client knows one address
print("🏆 BEST: API Gateway (centralized control)")
print(f"   URLs the client needs to know: 1")
print(f"   - http://localhost:8080")
print(f"   Load balancing: handled by gateway (automatic)")
print(f"   Auth/Rate limiting: centralized at gateway")
print()

# Quick latency comparison
def measure_avg_latency(url, n=20):
    times = []
    for _ in range(n):
        start = time.time()
        requests.get(url)
        times.append((time.time() - start) * 1000)
    return sum(times) / len(times)

direct_ms = measure_avg_latency("http://localhost:5001/users")
gateway_ms = measure_avg_latency("http://localhost:8080/api/users")

print(f"⏱️  Latency (avg of 20 requests):")
print(f"   Direct call:      {direct_ms:.1f} ms")
print(f"   Through gateway:  {gateway_ms:.1f} ms")
print(f"   Overhead:         ~{gateway_ms - direct_ms:.1f} ms")
print()
print("💡 The gateway adds a tiny bit of latency (usually <1ms) but gives you")
print("   routing, load balancing, auth, rate limiting, and more. Worth it!")

## 🧭 How Routing Works Under the Hood

When a request arrives at the gateway, nginx follows this process:

```
1. Client sends:  GET http://localhost:8080/api/users/1
                              │
2. nginx matches:  location /api/users  ← longest prefix match
                              │
3. nginx rewrites: /api/users/1  →  /users/1
                              │
4. nginx picks:    user-service-1:5000 (round-robin)
                              │
5. nginx forwards: GET http://user-service-1:5000/users/1
                              │
6. Backend responds → nginx returns response to client
```

The key config line is:
```nginx
location /api/users {
    proxy_pass http://user_backend/users;
}
```

- `location /api/users` — matches any URL starting with `/api/users`
- `proxy_pass http://user_backend/users` — forwards to the upstream, replacing `/api/users` with `/users`

## 📚 Summary

### What We Learned

| Approach | Entry Points | Load Balancing | Health Checks | Centralized Control |
|----------|-------------|----------------|---------------|--------------------|
| 🚫 BAD (direct) | Many (one per service) | Client-side | None | ❌ |
| ✅ BETTER (reverse proxy) | One | None | None | Partial |
| 🏆 BEST (API gateway) | One | Automatic | Automatic | ✅ |

### Key Takeaways

1. **Single entry point** — clients only need one URL, not one per service
2. **Path-based routing** — the gateway maps URL paths to backend services
3. **Load balancing** — traffic is automatically spread across service instances
4. **Health checks** — unhealthy backends are removed from rotation
5. **Minimal overhead** — the gateway adds <1ms of latency but huge operational benefits

### Interview Tip

> When drawing a system design, just say: *"I'll add an API Gateway to handle routing and basic middleware"* and draw a single box. Don't over-explain — the gateway is important but not the interesting part of most designs.

### Next Up

In **Notebook 2**, we'll add **rate limiting** and **API key authentication** at the gateway level — protecting all your services from one place.